# Día 19 — Práctica: Limpieza y Joins en pandas

**Dataset:** Empresa de servicios — `empleados.csv` + `proyectos.csv`  
**Instrucciones:** resuelve cada ejercicio sin ver el notebook de referencia. El resultado esperado está descrito en cada celda.

---

In [10]:
import pandas as pd

dfEmpleados = pd.read_csv('../data/day19/empleados.csv')
dfProyectos  = pd.read_csv('../data/day19/proyectos.csv')

print('Empleados:', dfEmpleados.shape)
print('Proyectos:', dfProyectos.shape)
print('--------------------------------')
print('Empleados:', dfEmpleados.dtypes)
print('________________________________')
print('Proyectos:', dfProyectos.dtypes)
print('--------------------------------')
print('Información estructurada sobre el dataset: ')
print('--------------------------------')
print('Empleados:', dfEmpleados.info())
print('________________________________')
print('Proyectos:', dfProyectos.info())
print('--------------------------------')
print('Primeros registros y nombre de cada campo: ')
print('--------------------------------')
print('Empleados:', dfEmpleados.head())
print('________________________________')
print('Proyectos:', dfProyectos.head())
print('--------------------------------')

Empleados: (50, 5)
Proyectos: (49, 6)
--------------------------------
Empleados: empleado_id      int64
nombre             str
departamento       str
salario            str
fecha_ingreso      str
dtype: object
________________________________
Proyectos: proyecto_id         int64
empleado_id         int64
cliente               str
presupuesto       float64
estado                str
duracion_meses    float64
dtype: object
--------------------------------
Información estructurada sobre el dataset: 
--------------------------------
<class 'pandas.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   empleado_id    50 non-null     int64
 1   nombre         50 non-null     str  
 2   departamento   44 non-null     str  
 3   salario        45 non-null     str  
 4   fecha_ingreso  42 non-null     str  
dtypes: int64(1), str(4)
memory usage: 2.1 KB
Empleados: None
__________________

---
## Ejercicio 1 — Detectar nulos

Calcula cuántos nulos hay por columna en `empleados` y qué porcentaje representa cada uno.

**Resultado esperado:** tabla con columnas `nulos` y `porcentaje_%`, mostrando solo las filas donde hay al menos un nulo.

In [11]:
# La identificación de nulos se con isnull(), que devuelve True donde hay NaN, y luego .sum() cuenta esos True
nulos = dfEmpleados.isnull().sum()

#Para saber el porcentaje de nulos se dicide la cantidad de nulos acumulada por el total de registros
ptcNulos = (nulos/ len(dfEmpleados)*100).round(2) 

#Se crea una tabla de muestreo con campos filtrados/nuevos 
# con un diccionario, donde cada clave equivale a una columna
resumen = pd.DataFrame({'nulos': nulos, 'porcentaje_%': ptcNulos})

# Se muestra el resumen filtrando unicamente por donde existen dentro de resumen
print(resumen[resumen['nulos']>0])

               nulos  porcentaje_%
departamento       6          12.0
salario            5          10.0
fecha_ingreso      8          16.0


---
## Ejercicio 2 — Tratar nulos

En `empleados`:
- Los nulos en `departamento` → rellenar con `'Sin asignar'`
- Los nulos en `salario` → eliminar la fila
- Los nulos en `fecha_ingreso` → eliminar la fila

**Resultado esperado:** imprimir cuántas filas se eliminaron y verificar que no quedan nulos en esas columnas.

In [12]:
# Departamento nulo equivale a rellenar campo nulo con valor asignado por fillna() = 'Sin asignar'
dfEmpleados['departamento'] = dfEmpleados['departamento'].fillna('Sin asignar')

filas_antes = len(dfEmpleados)

dfEmpleados = dfEmpleados.dropna(subset=['salario'])
dfEmpleados = dfEmpleados.dropna(subset=['fecha_ingreso'])

print('Filas eliminadas por salario o fecha nulos:', filas_antes - len(dfEmpleados))
print('Nulos restantes en salario, departamento y fecha de ingreso :')
print(dfEmpleados[['salario', 'departamento', 'fecha_ingreso']].isnull().sum())

Filas eliminadas por salario o fecha nulos: 13
Nulos restantes en salario, departamento y fecha de ingreso :
salario          0
departamento     0
fecha_ingreso    0
dtype: int64


---
## Ejercicio 3 — Eliminar duplicados

`empleados.csv` contiene filas duplicadas. Identifícalas, muestra cuáles son y elimínalas.

**Resultado esperado:** número de duplicados encontrados, detalle de los empleado_id duplicados, shape final tras eliminarlos.

In [13]:
print('Filas duplicadas: ',dfEmpleados.duplicated().sum())
print('Duplicados: ')
print(dfEmpleados[dfEmpleados.duplicated(keep=False)][['empleado_id', 'nombre']].sort_values('empleado_id'))
dfEmpleados = dfEmpleados.drop_duplicates()
print('Estructura o Shape final: ', dfEmpleados.shape)

Filas duplicadas:  3
Duplicados: 
    empleado_id          nombre
0           201    Ana Martínez
47          201    Ana Martínez
1           202  Carlos Sánchez
48          202  Carlos Sánchez
2           203    Laura García
49          203    Laura García
Estructura o Shape final:  (34, 5)


---
## Ejercicio 4 — Corregir tipos

Verifica los tipos de `empleados` con `dtypes`. Hay dos columnas con tipo incorrecto:
- `salario` está como texto (tiene el símbolo `€`) → convertir a `float`
- `fecha_ingreso` está como texto (formato `dd-mm-yyyy`) → convertir a `datetime`

**Resultado esperado:** mostrar `dtypes` antes y después de corregir.

In [14]:
print("Tipos actuales en 'Empleados': ")
print(dfEmpleados.dtypes)

# .str activa el modo texto para poder usar .replace() sobre cada valor de la columna
# sin .str, .replace() actúa sobre el DataFrame completo y no elimina el símbolo
dfEmpleados['salario'] = dfEmpleados['salario'].str.replace('€', '', regex=False).astype(float)
dfEmpleados['fecha_ingreso'] = pd.to_datetime(dfEmpleados['fecha_ingreso'], format='%d-%m-%Y')

print('Tipos corregidos:')
print(dfEmpleados[['fecha_ingreso', 'salario']].dtypes)
print(dfEmpleados[['fecha_ingreso', 'salario']].head(5))

Tipos actuales en 'Empleados': 
empleado_id      int64
nombre             str
departamento       str
salario            str
fecha_ingreso      str
dtype: object
Tipos corregidos:
fecha_ingreso    datetime64[us]
salario                 float64
dtype: object
  fecha_ingreso  salario
0    2019-03-15  42000.0
1    2018-07-22  55000.0
2    2020-01-10  38000.0
3    2017-06-05  65000.0
4    2021-09-18  44000.0


---
## Ejercicio 5 — INNER merge

Une `proyectos` con `empleados` usando la columna común `empleado_id`.

**Resultado esperado:** número de filas antes y después del merge. Primeras filas mostrando `proyecto_id`, `empleado_id`, `nombre`, `departamento`, `presupuesto`.

In [15]:
# pd.merge() une dos DataFrames por una columna que existe en ambos
# dfProyectos → DataFrame izquierdo (el principal: queremos ver todos los proyectos)
# dfEmpleados → DataFrame derecho (del que tomamos la información del empleado asignado)
# on='empleado_id' → columna puente que existe con ese nombre en los dos DataFrames
# how='inner' → devuelve solo los proyectos cuyo empleado_id existe en dfEmpleados
# los proyectos con empleado_id que no exista en dfEmpleados desaparecen del resultado
proyectos_detalle = pd.merge(dfProyectos, dfEmpleados, on='empleado_id', how='inner')

print('Proyectos originales: ', len(dfProyectos))
print('Resultado del merge:  ', len(proyectos_detalle))
print('Primeras filas:')
# seleccionamos solo esas columnas para que la salida sea legible
print(proyectos_detalle[['proyecto_id', 'empleado_id', 'nombre', 'departamento', 'presupuesto']].head())

Proyectos originales:  49
Resultado del merge:   34
Primeras filas:
   proyecto_id  empleado_id            nombre departamento  presupuesto
0            1          201      Ana Martínez       Ventas      15000.0
1            2          204     Miguel Torres           IT      32000.0
2            3          207  Carmen Fernández    Marketing       8500.0
3            6          216     Antonio López         RRHH      12000.0
4            7          219    Natalia Torres       Ventas      28000.0


---
## Ejercicio 6 — Comparar inner vs left merge

Haz los dos tipos de merge entre `proyectos` y `empleados`. Compara cuántas filas devuelve cada uno e identifica qué proyectos quedan fuera del inner merge.

**Resultado esperado:** número de filas en inner y left, lista de proyectos sin empleado registrado.

---
## Ejercicio 7 — Verificar integridad del merge

Después del INNER merge entre `proyectos` y `empleados`, verifica:
- ¿Se perdieron filas respecto a `proyectos` original?
- ¿Se duplicaron filas?
- ¿`proyecto_id` sigue siendo único en el resultado?

**Resultado esperado:** mensaje claro indicando si el merge fue correcto o si hubo pérdidas o duplicaciones.

In [16]:
print('Proyectos antes del merge: ', len(dfProyectos))
print('Proyectos después (inner): ', len(proyectos_detalle))

# restamos filas del merge menos filas originales para saber si se ganó o perdió algo
# diff > 0 → el merge añadió filas (hay empleado_ids repetidos en dfEmpleados)
# diff < 0 → el merge perdió filas (proyectos cuyo empleado no existe en dfEmpleados)
# diff = 0 → relación 1:1 perfecta
diff = len(proyectos_detalle) - len(dfProyectos)
if diff > 0:
    # un empleado_id duplicado en dfEmpleados multiplicaría los proyectos que lo referencian
    print('ALERTA: el merge añadió', diff, 'filas → hay duplicados en empleados')
elif diff < 0:
    # abs() convierte el número negativo en positivo para que el mensaje sea legible
    print('INFO: se perdieron', abs(diff), 'proyectos → no tienen empleado registrado')
else:
    print('OK: merge 1:1, sin pérdidas ni duplicaciones')

# verificación final: proyecto_id debe seguir siendo único en el resultado
# si .duplicated().sum() devuelve 0, cada proyecto aparece solo una vez → merge correcto
duplicados_id = proyectos_detalle['proyecto_id'].duplicated().sum()
print('proyecto_ids duplicados post-merge:', duplicados_id)

Proyectos antes del merge:  49
Proyectos después (inner):  34
INFO: se perdieron 15 proyectos → no tienen empleado registrado
proyecto_ids duplicados post-merge: 0
